# 02 - Data Preprocessing

## Đề tài
Dự đoán chất lượng rượu vang bằng hồi quy

## Thành viên
- Nguyễn Văn Vũ - 12523093
- Nguyễn Văn Linh - 10123207

## Mục tiêu

Thực hiện tiền xử lý dữ liệu Wine Quality trước khi huấn luyện
các mô hình hồi quy.

Các công việc chính:
1. Đọc dữ liệu
2. Kiểm tra dữ liệu
3. Xử lý dữ liệu trùng
4. Xác định biến đầu vào X và biến mục tiêu y
5. Chia dữ liệu thành tập train và test
6. Xây dựng pipeline tiền xử lý
7. Tạo schema.json

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import os
import json

In [4]:
df = pd.read_csv("/content/winequality-red.csv")

print("Kích thước dataset:", df.shape)

df.head()

Kích thước dataset: (1599, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [5]:
print("=== THÔNG TIN DATASET ===")
df.info()

print("\n=== DỮ LIỆU THIẾU ===")
print(df.isnull().sum())

print("\n=== DỮ LIỆU TRÙNG ===")
print("Số dòng trùng:", df.duplicated().sum())

=== THÔNG TIN DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB

=== DỮ LIỆU THIẾU ===
fixed acidity           0
volatile acidity        0
citric acid             0

In [6]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Số dòng trước khi xóa trùng:", before)
print("Số dòng sau khi xóa trùng:", after)
print("Số dòng đã loại bỏ:", before - after)

Số dòng trước khi xóa trùng: 1599
Số dòng sau khi xóa trùng: 1359
Số dòng đã loại bỏ: 240


In [7]:
print("Số dòng trùng còn lại:", df.duplicated().sum())

Số dòng trùng còn lại: 0


In [8]:
print("Kích thước dataset sau khi xử lý:", df.shape)

df.describe().T

Kích thước dataset sau khi xử lý: (1359, 12)


,count,mean,std,min,25%,50%,75%,max
fixed acidity,1359.0,8.310596,1.736990,4.60000,7.1000,7.9000,9.20000,15.90000
volatile acidity,1359.0,0.529478,0.183031,0.12000,0.3900,0.5200,0.64000,1.58000
citric acid,1359.0,0.272333,0.195537,0.00000,0.0900,0.2600,0.43000,1.00000
residual sugar,1359.0,2.523400,1.352314,0.90000,1.9000,2.2000,2.60000,15.50000
chlorides,1359.0,0.088124,0.049377,0.01200,0.0700,0.0790,0.09100,0.61100
free sulfur dioxide,1359.0,15.893304,10.447270,1.00000,7.0000,14.0000,21.00000,72.00000
total sulfur dioxide,1359.0,46.825975,33.408946,6.00000,22.0000,38.0000,63.00000,289.00000
density,1359.0,0.996709,0.001869,0.99007,0.9956,0.9967,0.99782,1.00369
pH,1359.0,3.309787,0.155036,2.74000,3.2100,3.3100,3.40000,4.01000
sulphates,1359.0,0.658705,0.170667,0.33000,0.5500,0.6200,0.73000,2.00000


In [9]:
X = df.drop("quality", axis=1)
y = df["quality"]

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)

print("\nCác biến đầu vào:")
print(X.columns.tolist())

print("\nBiến mục tiêu:")
print(y.name)

Kích thước X: (1359, 11)
Kích thước y: (1359,)

Các biến đầu vào:
['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']

Biến mục tiêu:
quality


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (1087, 11)
X_test : (272, 11)
y_train: (1087,)
y_test : (272,)


In [11]:
preprocessing_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [12]:
preprocessing_pipeline.fit(X_train)

X_train_processed = preprocessing_pipeline.transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print("X_train sau preprocessing:", X_train_processed.shape)
print("X_test sau preprocessing:", X_test_processed.shape)

X_train sau preprocessing: (1087, 11)
X_test sau preprocessing: (272, 11)


In [13]:
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
865,8.9,0.38,0.40,2.2,0.068,12.0,28.0,0.99486,3.27,0.75,12.6
1289,6.6,0.70,0.08,2.6,0.106,14.0,27.0,0.99665,3.44,0.58,10.2
394,6.8,0.56,0.03,1.7,0.084,18.0,35.0,0.99680,3.44,0.63,10.0
731,7.4,0.68,0.16,1.8,0.078,12.0,39.0,0.99770,3.50,0.70,9.9
54,7.8,0.59,0.18,2.3,0.076,17.0,54.0,0.99750,3.43,0.59,10.0
...,...,...,...,...,...,...,...,...,...,...,...
1095,11.3,0.37,0.50,1.8,0.090,20.0,47.0,0.99734,3.15,0.57,10.5
1130,7.4,0.60,0.26,2.1,0.083,17.0,91.0,0.99616,3.29,0.56,9.8
1294,6.8,0.47,0.08,2.2,0.064,18.0,38.0,0.99553,3.30,0.65,9.6
860,8.9,0.32,0.31,2.0,0.088,12.0,19.0,0.99570,3.17,0.55,10.4


In [14]:
print("Mean của các feature sau scaling:")
print(X_train_processed.mean(axis=0))

print("\nStandard deviation:")
print(X_train_processed.std(axis=0))

Mean của các feature sau scaling:
[ 3.57886061e-16 -1.14392805e-16 -4.57571219e-17 -2.61469268e-17
 -5.22938536e-17 -1.47076463e-17  1.30734634e-17 -9.97505257e-15
 -3.01997005e-15 -4.11814097e-16 -8.90629694e-16]

Standard deviation:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [15]:
os.makedirs("/content/models", exist_ok=True)

In [16]:
import joblib

pipeline_path = "/content/models/preprocessing_pipeline.joblib"

joblib.dump(preprocessing_pipeline, pipeline_path)

print("Đã lưu pipeline tại:")
print(pipeline_path)

Đã lưu pipeline tại:
/content/models/preprocessing_pipeline.joblib


In [17]:
schema = {
    "problem_type": "regression",
    "target": {
        "name": "quality",
        "type": "integer",
        "min": int(y.min()),
        "max": int(y.max())
    },
    "features": []
}

for column in X.columns:
    schema["features"].append({
        "name": column,
        "type": str(X[column].dtype),
        "min": float(X[column].min()),
        "max": float(X[column].max())
    })

schema

{'problem_type': 'regression',
 'target': {'name': 'quality', 'type': 'integer', 'min': 3, 'max': 8},
 'features': [{'name': 'fixed acidity',
   'type': 'float64',
   'min': 4.6,
   'max': 15.9},
  {'name': 'volatile acidity', 'type': 'float64', 'min': 0.12, 'max': 1.58},
  {'name': 'citric acid', 'type': 'float64', 'min': 0.0, 'max': 1.0},
  {'name': 'residual sugar', 'type': 'float64', 'min': 0.9, 'max': 15.5},
  {'name': 'chlorides', 'type': 'float64', 'min': 0.012, 'max': 0.611},
  {'name': 'free sulfur dioxide', 'type': 'float64', 'min': 1.0, 'max': 72.0},
  {'name': 'total sulfur dioxide',
   'type': 'float64',
   'min': 6.0,
   'max': 289.0},
  {'name': 'density', 'type': 'float64', 'min': 0.99007, 'max': 1.00369},
  {'name': 'pH', 'type': 'float64', 'min': 2.74, 'max': 4.01},
  {'name': 'sulphates', 'type': 'float64', 'min': 0.33, 'max': 2.0},
  {'name': 'alcohol', 'type': 'float64', 'min': 8.4, 'max': 14.9}]}

In [18]:
schema_path = "/content/models/schema.json"

with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4, ensure_ascii=False)

print("Đã lưu:", schema_path)

Đã lưu: /content/models/schema.json


In [19]:
with open("/content/models/schema.json", "r", encoding="utf-8") as f:
    print(f.read())

{
    "problem_type": "regression",
    "target": {
        "name": "quality",
        "type": "integer",
        "min": 3,
        "max": 8
    },
    "features": [
        {
            "name": "fixed acidity",
            "type": "float64",
            "min": 4.6,
            "max": 15.9
        },
        {
            "name": "volatile acidity",
            "type": "float64",
            "min": 0.12,
            "max": 1.58
        },
        {
            "name": "citric acid",
            "type": "float64",
            "min": 0.0,
            "max": 1.0
        },
        {
            "name": "residual sugar",
            "type": "float64",
            "min": 0.9,
            "max": 15.5
        },
        {
            "name": "chlorides",
            "type": "float64",
            "min": 0.012,
            "max": 0.611
        },
        {
            "name": "free sulfur dioxide",
            "type": "float64",
            "min": 1.0,
            "max": 72.0
        },
     

In [20]:
print("========== TỔNG KẾT PREPROCESSING ==========")

print("Dataset sau xử lý:", df.shape)
print("Số feature:", X.shape[1])
print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nTarget:", y.name)
print("Target min:", y.min())
print("Target max:", y.max())

print("\nDuplicate còn lại:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum())

print("\nPipeline:", pipeline_path)
print("Schema:", schema_path)

========== TỔNG KẾT PREPROCESSING ==========
Dataset sau xử lý: (1359, 12)
Số feature: 11
Train: (1087, 11)
Test : (272, 11)

Target: quality
Target min: 3
Target max: 8

Duplicate còn lại: 0
Missing values: 0

Pipeline: /content/models/preprocessing_pipeline.joblib
Schema: /content/models/schema.json


In [22]:
import os
import joblib

os.makedirs("/content/models", exist_ok=True)

joblib.dump(
    preprocessing_pipeline,
    "/content/models/preprocessing_pipeline.joblib"
)

print("Đã lưu preprocessing pipeline")

Đã lưu preprocessing pipeline
